In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

X_full = pd.read_csv("/kaggle/input/titanic/train.csv")
X_test  = pd.read_csv("/kaggle/input/titanic/test.csv")

y = X_full.Survived
X_train = X_full.drop(["Survived", "Name"], axis=1).copy()
X_test  = X_test.drop(["Name"], axis=1).copy()
print(X_train.Age.isnull().any())
# Define bins for ages (note: bins here do NOT include a "missing" interval;
# missing ages will be set to 1 afterwards)
bins   = [0, 5, 12, 18, 35, 60, 100]         # 7-1 = 6 intervals
labels = [2, 3, 4, 5, 6, 7]                 # numeric labels for the 6 real intervals

# Apply to train
X_train['AgeGroup'] = pd.cut(X_train['Age'], bins=bins, labels=labels, right=False)
# convert to numeric and set missing Age -> code 1
X_train['AgeGroup'] = X_train['AgeGroup'].astype(float)

X_train['AgeGroup'] = X_train['AgeGroup'].astype(int)

# Drop original Age column (assign back)
X_train = X_train.drop(["Age"], axis=1)

# Apply same to test
X_test['AgeGroup'] = pd.cut(X_test['Age'], bins=bins, labels=labels, right=False)
X_test['AgeGroup'] = X_test['AgeGroup'].astype(float)
 # note: use X_test here
X_test['AgeGroup'] = X_test['AgeGroup'].astype(int)
X_test = X_test.drop(["Age"], axis=1)
X_train.AgeGroup.isnull().any()
# quick check



In [ ]:
from sklearn.pipeline import Pipeline
from xgboost import XGBClassifier
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder


imputer=SimpleImputer(strategy="constant")
cat_transform = Pipeline(steps=[
    ('encode', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)),
    ('impute', KNNImputer(n_neighbors=5))
])


In [ ]:
cat_cols=[cname for cname in X_train.columns if X_train[cname].nunique() < 10 and  X_train[cname].dtype == "object"]
numerical_cols = [cname for cname in X_train.columns if X_train[cname].dtype in ['int64', 'float64']]

In [ ]:
from sklearn.compose import ColumnTransformer
numerical_transformer =  KNNImputer(n_neighbors=5)
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numerical_transformer, numerical_cols),
        ('cat', cat_transform, cat_cols)
    ])
X_train.head()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error
model = RandomForestClassifier(max_leaf_nodes=100)
my_pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                              ('model', model)
                             ])
my_pipeline.fit(X_train, y)


In [ ]:

x=my_pipeline.predict(X_test)

In [ ]:
output = pd.DataFrame({'PassengerId': X_test.PassengerId,
                       'Survived': x})
output.to_csv('submission.csv', index=False)